In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import json

# ==========================================
# Event Hub Connection
# ==========================================

connectionString = (
    "Endpoint=sb://eh-social-media.servicebus.windows.net/;"
    "SharedAccessKeyName=sudheer;"
    "SharedAccessKey=1IhrynvVGzNbhQLyvqMUxuT7QdTRHRh57+AEhJK7+eQ=;"
    "EntityPath=trends-hub"
)

ehConf = {}

ehConf["eventhubs.connectionString"] = (
    sc._jvm.org.apache.spark.eventhubs.EventHubsUtils.encrypt(connectionString)
)

ehConf["eventhubs.consumerGroup"] = "$Default"

startingEventPosition = {
    "offset": "-1",
    "seqNo": -1,
    "enqueuedTime": None,
    "isInclusive": True
}

ehConf["eventhubs.startingPosition"] = json.dumps(startingEventPosition)
ehConf["eventhubs.maxEventsPerTrigger"] = "5000"

# ==========================================
# Read Stream
# ==========================================

rawDF = (
    spark.readStream
         .format("eventhubs")
         .options(**ehConf)
         .load()
)

# Convert binary body to string
eventDF = rawDF.selectExpr(
    "CAST(body AS STRING) AS message"
)

# ==========================================
# Schema
# ==========================================

schema = StructType([
    StructField("trend_timestamp", StringType(), True),
    StructField("topic_category", StringType(), True),
    StructField("country", StringType(), True),
    StructField("tweet_volume", DoubleType(), True),
    StructField("mention_count", DoubleType(), True),
    StructField("retweet_count", DoubleType(), True),
    StructField("trend_score", DoubleType(), True),
    StructField("sentiment_index", DoubleType(), True),
    StructField("impressions", DoubleType(), True),
    StructField("engagement_count", DoubleType(), True)
])

# ==========================================
# Parse JSON
# ==========================================

parsedDF = (
    eventDF
        .select(from_json(col("message"), schema).alias("data"))
        .select("data.*")
)

# ==========================================
# Error Records
# ==========================================

errorDF = parsedDF.filter(col("trend_timestamp").isNull())

errorQuery = (
    errorDF.writeStream
        .trigger(availableNow=True)
        .format("delta")
        .outputMode("append")
        .option(
            "checkpointLocation",
            "abfss://raw@socialmedia1.dfs.core.windows.net/checkpoints/error_bronze_trends"
        )
        .toTable("socialmedia_databricks.error.error_bronze_trends")
)

# ==========================================
# Bronze Data
# ==========================================

bronzeDF = (
    parsedDF
        .filter(col("trend_timestamp").isNotNull())
        .withColumn(
        "trend_timestamp",
        to_timestamp(col("trend_timestamp"), "dd-MM-yyyy")
    )
        .withColumn("bronze_load_time", current_timestamp())
        .withColumn("pipeline_name", lit("Bronze_Trends"))
        .withColumn("source_system", lit("Azure Event Hub"))
        .withColumn("ingestion_date", current_date())
        .withWatermark("trend_timestamp", "10 minutes")
)

# ==========================================
# Write Bronze Table
# ==========================================

bronzeQuery = (
    bronzeDF.writeStream
        .trigger(availableNow=True)
        .format("delta")
        .outputMode("append")
        .option(
            "checkpointLocation",
            "abfss://raw@socialmedia1.dfs.core.windows.net/checkpoints/bronze_trends"
        )
        .option("mergeSchema", "true")
        .toTable("socialmedia_databricks.bronze.bronze_trends")
)

bronzeQuery.awaitTermination()
errorQuery.awaitTermination()

print("Bronze Trends Loaded Successfully")

Bronze Trends Loaded Successfully


In [0]:
%sql
select * from socialmedia_databricks.bronze.bronze_trends;

trend_timestamp,topic_category,country,tweet_volume,mention_count,retweet_count,trend_score,sentiment_index,impressions,engagement_count,bronze_load_time,pipeline_name,source_system,ingestion_date
2025-01-22T00:00:00Z,AI,India,2265.0,6221.0,2688.0,0.556363715,0.865507353,14572.0,15986.0,2026-07-09T05:30:21.293Z,Bronze_Trends,Azure Event Hub,2026-07-09
2025-01-17T00:00:00Z,AI,USA,7341.0,7102.0,1473.0,-0.615185751,0.764488934,14616.0,8781.0,2026-07-09T05:30:21.293Z,Bronze_Trends,Azure Event Hub,2026-07-09
2025-01-21T00:00:00Z,Sports,USA,6294.0,NaN,9095.0,-0.145476515,0.637842072,11954.0,20744.0,2026-07-09T05:30:21.293Z,Bronze_Trends,Azure Event Hub,2026-07-09
2025-01-15T00:00:00Z,AI,Germany,1645.0,5702.0,8746.0,0.931839294,0.490921453,31204.0,22183.0,2026-07-09T05:30:21.293Z,Bronze_Trends,Azure Event Hub,2026-07-09
2025-01-20T00:00:00Z,AI,India,4694.0,6548.0,924.0,NaN,0.327932839,84641.0,48787.0,2026-07-09T05:30:21.293Z,Bronze_Trends,Azure Event Hub,2026-07-09
2025-01-23T00:00:00Z,Sports,UK,4620.0,7420.0,444.0,-0.47045313,0.598963113,90377.0,48063.0,2026-07-09T05:30:21.293Z,Bronze_Trends,Azure Event Hub,2026-07-09
2025-01-15T00:00:00Z,Sports,India,5087.0,7387.0,2361.0,0.750328729,0.480904853,58734.0,37846.0,2026-07-09T05:30:21.293Z,Bronze_Trends,Azure Event Hub,2026-07-09
2025-01-11T00:00:00Z,AI,UK,4711.0,3978.0,9530.0,-0.817703797,0.16200097,96093.0,16096.0,2026-07-09T05:30:21.293Z,Bronze_Trends,Azure Event Hub,2026-07-09
2025-01-15T00:00:00Z,AI,India,6972.0,3705.0,523.0,-0.909454978,NaN,24826.0,26466.0,2026-07-09T05:30:21.293Z,Bronze_Trends,Azure Event Hub,2026-07-09
2025-01-15T00:00:00Z,Cloud,Germany,6966.0,2984.0,2714.0,-0.568140431,0.614473314,81784.0,28572.0,2026-07-09T05:30:21.293Z,Bronze_Trends,Azure Event Hub,2026-07-09
